# Lab: Uniswap V2 AMM — Implementation & Analysis

**Prerequisites:** Python 3.10+, `matplotlib`, `numpy`  
**Estimated time:** 2 hours

---

## Introduction

Most DeFi protocols are built on top of one core primitive: the **Automated Market Maker (AMM)**. Instead of matching buyers and sellers through an order book, an AMM uses a mathematical formula to determine prices algorithmically.

Uniswap V2 uses the **constant product formula**:

$$x \cdot y = k$$

Where `x` and `y` are the reserves of two tokens in a pool, and `k` is a constant that must hold after every trade (accounting for fees). This deceptively simple equation encodes all the pool's behavior: pricing, slippage, liquidity incentives.

In this lab you will:
1. Implement a working AMM pool from scratch
2. Analyze how trade size affects price
3. Model liquidity provision and fee accrual

In [1]:
import math
import random

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

---

## Part 1 — Pool Mechanics

### 1.1 The `UniswapV2Pool` Class

Implement the pool. The constructor takes initial reserves and a fee in **basis points** (1 bps = 0.01%, so the default 30 bps = 0.3%).

**Task 1.1** — Implement all properties. Verify with a sanity check: pool with `reserve_x=1000, reserve_y=2_000_000` should have `price = 2000.0` and `k = 2_000_000_000`.

In [2]:
class UniswapV2Pool:
    def __init__(self, reserve_x: float, reserve_y: float, fee_bps: int = 30):
        self.reserve_x = reserve_x
        self.reserve_y = reserve_y
        self.fee_bps = fee_bps
        self.total_shares = 0.0
        self.lp_positions = {}
        self.lp_entry_k = {}
        

    @property
    def k(self) -> float:
        """Constant product invariant x * y."""
        return self.reserve_x * self.reserve_y

    @property
    def price(self) -> float:
        """Spot price of X denominated in Y (e.g. ETH price in USDC)."""
        return self.reserve_y / self.reserve_x

    @property
    def fee_rate(self) -> float:
        """Fee as a decimal fraction (e.g. 0.003 for 30 bps)."""
        return self.fee_bps / 10000

    def __repr__(self) -> str:
        return f"UniswapV2Pool(reserve_x={self.reserve_x}, reserve_y={self.reserve_y}, price={self.price}, k={self.k})"

    # ----- Methods below will be implemented in later tasks -----

    def get_amount_out(self, amount_in: float, token: str) -> float:
        """
        Pure calculation — does NOT change state.
        token='x' means selling X to receive Y, and vice versa.
        """
        assert token in ['x', 'y'], "token must be 'x' or 'y'"

        x = self.reserve_x
        y = self.reserve_y
        amount_in_with_fee = amount_in * (1 - self.fee_rate)
        if token == 'x':
            amount_out = (amount_in_with_fee * y) / (x + amount_in_with_fee)
        else:
            amount_out = (amount_in_with_fee * x) / (y + amount_in_with_fee)
        return amount_out


    def swap(self, amount_in: float, token: str) -> float:
        """
        Executes a swap. Updates reserves and returns amount_out.
        """
        k_before = self.k
        amount_out = self.get_amount_out(amount_in, token)
        
        if token == 'x':
            self.reserve_x += amount_in
            self.reserve_y -= amount_out
        else:
            self.reserve_y += amount_in
            self.reserve_x -= amount_out

        assert self.k >= k_before, "Invariant k decreased after swap"

        return amount_out

    def add_liquidity(self, amount_x: float, amount_y: float, provider: str) -> float:
        """
        Adds liquidity. For the first deposit uses sqrt(x * y).
        For subsequent deposits, adjusts amount_y to match the current ratio.
        Returns number of shares minted.
        """
        if self.total_shares == 0:
            self.reserve_x = amount_x
            self.reserve_y = amount_y
            self.total_shares = math.sqrt(amount_x * amount_y)
        else:
            amount_y = amount_x * (self.reserve_y / self.reserve_x)
            shares = (amount_x / self.reserve_x) * self.total_shares
            self.reserve_x += amount_x
            self.reserve_y += amount_y
            self.total_shares += shares

        shares = self.total_shares if self.total_shares == math.sqrt(amount_x * amount_y) else shares
        
        if provider not in self.lp_positions:
            self.lp_positions[provider] = 0.0
        self.lp_positions[provider] += shares
        self.lp_entry_k[provider] = self.k
        return shares

    def remove_liquidity(self, shares: float, provider: str) -> tuple[float, float]:
        """
        Burns shares and returns (amount_x, amount_y) to the provider.
        """
        assert provider in self.lp_positions, "Unknown provider"
        assert self.lp_positions[provider] >= shares, "Insufficient shares"

        ownership = shares / self.total_shares
        amount_x = ownership * self.reserve_x
        amount_y = ownership * self.reserve_y
        
        self.reserve_x -= amount_x
        self.reserve_y -= amount_y
        self.total_shares -= shares
        self.lp_positions[provider] -= shares
        
        return amount_x, amount_y

    def fees_earned(self, provider: str) -> dict:
        """
        Estimates fees earned since deposit based on k growth.
        Returns a dict with deposit_x, deposit_y, current_x, current_y, fee_x, fee_y.
        """
        assert provider in self.lp_positions

        if self.total_shares == 0 or self.lp_positions[provider] == 0:
            return {'deposit_x': 0, 'deposit_y': 0, 'current_x': 0, 'current_y': 0, 'fee_x': 0, 'fee_y': 0}

        ownership = self.lp_positions[provider] / self.total_shares
        current_x = ownership * self.reserve_x
        current_y = ownership * self.reserve_y
        
        entry_k = self.lp_entry_k.get(provider, self.k)
        k_ratio = math.sqrt(self.k / entry_k) if entry_k > 0 else 1.0
        
        deposit_x = current_x / k_ratio
        deposit_y = current_y / k_ratio
        
        fee_x = current_x - deposit_x
        fee_y = current_y - deposit_y
        
        return {
            'deposit_x': deposit_x,
            'deposit_y': deposit_y,
            'current_x': current_x,
            'current_y': current_y,
            'fee_x': fee_x,
            'fee_y': fee_y
        }

#### Sanity check — Task 1.1

Create a pool with `reserve_x=1000, reserve_y=2_000_000` and verify `price == 2000.0` and `k == 2_000_000_000`.

In [ ]:
# create pool and print price, k, and the pool repr
pool = UniswapV2Pool(reserve_x=1000, reserve_y=2_000_000)
print(f"Price: {pool.price}")
print(f"k: {pool.k}")
print(pool)


---

### 1.2 Computing `get_amount_out`

The formula accounts for the fee taken on the input side:

$$\text{amount\_in\_with\_fee} = \text{amount\_in} \times (1 - \text{fee\_rate})$$

$$\text{amount\_out} = \frac{\text{reserve\_out} \times \text{amount\_in\_with\_fee}}{\text{reserve\_in} + \text{amount\_in\_with\_fee}}$$

This is derived directly from $x \cdot y = k$ — solve for the new reserve after the swap, take the difference. The fee is *not* returned to the sender; it stays in the pool.

**Task 1.2** — Go back to the class cell above and implement `get_amount_out`. Then verify by hand: pool is `100 X / 100 Y` (price = 1.0), fee = 0%, swap 10 X. You should receive exactly `9.09...` Y, not 10. Why?

In [ ]:
# TODO: create a pool with 100 X / 100 Y, fee_bps=0
# TODO: compute get_amount_out(10, 'x') and verify it equals ~9.0909
# TODO: explain in a comment why the output is less than 10
Pool1 = UniswapV2Pool(100, 100, fee_bps=0)
amount_out = Pool1.get_amount_out(10, 'x')
print(f"Amount out for selling 10 X: {amount_out:.4f}")

---

### 1.3 Executing a Swap

**Task 1.3** — Go back to the class cell above and implement `swap`. Then run 1000 random swaps and assert `k` never decreases. Print `k` before and after the full run — does it grow? Why does it sometimes *increase*? Where do the fees go?

In [ ]:
pool = UniswapV2Pool(1000, 2_000_000, fee_bps=30)
k_initial = pool.k
for _ in range(1000):
    token = random.choice(['x', 'y'])
    if token == 'x':
        amount = random.uniform(0, 0.02 * pool.reserve_x)
    else:
        amount = random.uniform(0, 0.02 * pool.reserve_y)
    
    k_before = pool.k
    pool.swap(amount, token)
    assert pool.k >= k_before, "k decreased"

k_final = pool.k
print(f"k_initial: {k_initial}")
print(f"k_final: {k_final}")

*Your explanation (why does k grow?):*

YOUR ANSWER HERE

---

## Part 2 — Price Impact

Price impact is the difference between the price you *expected* (spot price before the trade) and the price you *actually got* (execution price).

$$\text{price\_impact} = \frac{p_{\text{spot}} - p_{\text{execution}}}{p_{\text{spot}}}$$

For a swap X → Y:
- Spot price: `reserve_y / reserve_x`
- Execution price: `amount_out / amount_in`

### 2.1 Single Swap Analysis

**Task 2.1** — Compute spot price, execution price, and price impact for `amount_in = 1, 10, 50, 100` ETH. Fill in the table below.

In [ ]:
pool = UniswapV2Pool(reserve_x=1000, reserve_y=2_000_000)  # 1 ETH = 2000 USDC

for amount_in in [1, 10, 50, 100]:
    spot_price = pool.price
    amount_out = pool.get_amount_out(amount_in, 'x')
    execution_price = amount_out / amount_in
    price_impact = (spot_price - execution_price) / spot_price
    print(f"Amount In: {amount_in} ETH | Spot Price: {spot_price:.2f} | Execution Price: {execution_price:.2f} | Price Impact: {price_impact*100:.2f}%")

**Fill in the table:**

| Amount In (ETH) | Spot Price | Exec Price | Price Impact |
|---|---|---|---|
| 1 | | | |
| 10 | | | |
| 50 | | | |
| 100 | | | |

---

### 2.2 Experiment: Swap Size vs Price Impact

Sweep `amount_in` from 0.1 ETH to 500 ETH and plot the price impact curve.

**Task 2.2** — Create the plot with:
- Price impact (%) on y-axis vs trade size as % of `reserve_x` on x-axis
- Horizontal dashed lines at 1% and 5% impact
- Axis labels, title, legend, grid

In [ ]:
pool = UniswapV2Pool(reserve_x=1000, reserve_y=2_000_000)
sizes = np.linspace(0.1, 500, 300)
impacts = []

for size in sizes:
    spot_price = pool.price
    amount_out = pool.get_amount_out(size, 'x')
    execution_price = amount_out / size
    price_impact = (spot_price - execution_price) / spot_price
    impacts.append(price_impact * 100)

plt.plot(sizes / pool.reserve_x * 100, impacts, label='Price Impact Curve')
plt.axhline(y=1.0, color='r', linestyle='--', label='1% Impact')
plt.axhline(y=5.0, color='g', linestyle='--', label='5% Impact')
plt.xlabel('Trade Size as % of reserve_x')
plt.ylabel('Price Impact (%)')
plt.title('Swap Size vs Price Impact')
plt.legend()
plt.grid(True)
plt.show()

**Research questions (Task 2.2):**
- At what trade size (as % of pool) does price impact cross 1%? Cross 5%?
- Is the curve linear or convex? Explain why from the formula.
- At 10% of pool size, what is the approximate impact? Is this acceptable for a DEX user?

YOUR ANSWER HERE

---

### 2.3 Experiment: Pool Depth vs Price Impact

Fix the trade at \$10,000. Vary pool TVL from \$100k to \$100M and observe how pool depth affects price.

**Task 2.3** — Create the plot with:
- Price impact (%) on y-axis vs TVL (\$) on x-axis with log scale on x
- Horizontal dashed lines at 0.1% and 1.0%
- Axis labels, title, legend, grid

In [ ]:
trade_size_usd = 10_000
price_eth = 2000
tvl_values = np.logspace(5, 8, 100)  # $100k -> $100M on log scale

impacts = []

for tvl in tvl_values:
    reserve_y = tvl / 2
    reserve_x = reserve_y / price_eth
    pool = UniswapV2Pool(reserve_x, reserve_y)
    
    amount_in = trade_size_usd / price_eth
    spot_price = pool.price
    amount_out = pool.get_amount_out(amount_in, 'x')
    execution_price = amount_out / amount_in
    price_impact = (spot_price - execution_price) / spot_price
    
    impacts.append(price_impact * 100)

plt.plot(tvl_values, impacts, label='Price Impact curve ($10k Trade)')
plt.axhline(y=0.1, color='r', linestyle='--', label='0.1% Impact')
plt.axhline(y=1.0, color='g', linestyle='--', label='1.0% Impact')
plt.xscale('log')
plt.xlabel('TVL ($)')
plt.ylabel('Price Impact (%)')
plt.title('Pool Depth vs Price Impact for a $10k Trade')
plt.legend()
plt.grid(True)
plt.show()

**Research questions (Task 2.3):**
- What minimum TVL keeps impact below 0.1% for a \$10k trade?
- If trade size doubles, how does required TVL change?
- Real Uniswap V2 ETH/USDC had ~\$300M TVL at peak. Estimate the impact for a \$1M trade.

YOUR ANSWER HERE

---

### 2.4 Bonus: Comparing Fee Tiers

**Task 2.4** — Plot price impact for three fee tiers (0.05%, 0.3%, 1.0%) on the same axes. At what trade size does the fee start to dominate slippage as the main cost?

In [ ]:
fee_tiers = [5, 30, 100]   # 0.05%, 0.3%, 1.0%
sizes = np.linspace(0.1, 300, 200)

for fee in fee_tiers:
    pool = UniswapV2Pool(1000, 2_000_000, fee_bps=fee)
    impacts = []
    for size in sizes:
        spot_price = pool.price
        amount_out = pool.get_amount_out(size, 'x')
        execution_price = amount_out / size
        price_impact = (spot_price - execution_price) / spot_price
        impacts.append(price_impact * 100)
    plt.plot(sizes, impacts, label=f"{fee / 100}% Fee")

plt.xlabel('Trade Size (ETH)')
plt.ylabel('Price Impact (%)')
plt.title('Price Impact across Fee Tiers')
plt.legend()
plt.grid(True)
plt.show()

**Research question (Task 2.4):**

At what trade size does the fee start to dominate slippage as the main cost? What does this mean for a trader choosing between pools?

YOUR ANSWER HERE

---

## Part 3 — Liquidity Provision

Liquidity providers (LPs) deposit both tokens into the pool and receive **LP shares** representing their proportional ownership. As trades happen, fees accumulate in the pool — LPs profit when they withdraw.

### 3.1 Adding Liquidity

The key constraint: **you must deposit tokens in the same ratio as the current pool**, otherwise you'd instantly shift the price.

$$\text{shares} = \frac{\Delta x}{x} \times S_{\text{total}}$$

For the very first deposit, shares are initialized as $\sqrt{x \cdot y}$ (Uniswap convention — makes the initial share count independent of price).

**Task 3.1** — Go back to the class cell and implement `add_liquidity`. Then:
- Add 100 ETH / 200,000 USDC (price = 2000). Print shares minted and ownership %.
- Try adding with a wrong ratio (100 ETH / 100,000 USDC) — what does your implementation do, and why is this correct?

In [ ]:
pool = UniswapV2Pool(0, 0)
shares_minted = pool.add_liquidity(100, 200_000, "Alice")
ownership = shares_minted / pool.total_shares
print(f"Shares Minted: {shares_minted}, Ownership: {ownership * 100}%")

try:
    pool.add_liquidity(100, 100_000, "Bob")
except Exception as e:
    print(f"Error adding liquidity: {e}")
print(f"Bob actually added with matched ratio, y taken: {100 * (pool.reserve_y / pool.reserve_x)}")

---

### 3.2 Removing Liquidity

**Task 3.2** — Go back to the class cell and implement `remove_liquidity`. Then:
- Deposit as "Alice", simulate 100 random swaps, then remove all of Alice's liquidity.
- Compare what she deposited vs what she receives. Where did the difference come from?

In [ ]:
pool = UniswapV2Pool(0, 0)
shares = pool.add_liquidity(100, 200_000, "Alice")
deposit_x, deposit_y = 100, 200_000

for _ in range(100):
    token = random.choice(['x', 'y'])
    if token == 'x':
        amount = random.uniform(0, 0.02 * pool.reserve_x)
    else:
        amount = random.uniform(0, 0.02 * pool.reserve_y)
    pool.swap(amount, token)

withdrawn_x, withdrawn_y = pool.remove_liquidity(shares, "Alice")
diff_x = withdrawn_x - deposit_x
diff_y = withdrawn_y - deposit_y

print(f"Deposited: {deposit_x} X, {deposit_y} Y")
print(f"Withdrawn: {withdrawn_x} X, {withdrawn_y} Y")
print(f"Difference: {diff_x} X, {diff_y} Y")

*Your explanation (where did the difference come from?):*

YOUR ANSWER HERE

---

### 3.3 Fee Accrual

Fees are not paid out directly — they stay in the pool as increased reserves, so `k` grows over time. When an LP withdraws, their share is worth more than at deposit. The growth factor is $\sqrt{k_{\text{current}} / k_{\text{entry}}}$ because both reserves grow symmetrically.

**Task 3.3** — Go back to the class cell and implement `fees_earned`. Then:
- Verify `fees_earned` makes sense: deposit → 0 swaps → fees should be ~0.
- Then do 1000 swaps and check again. Do fees grow monotonically with number of swaps?

In [ ]:
pool = UniswapV2Pool(0, 0)
shares = pool.add_liquidity(100, 200_000, "Alice")
fees_earned = pool.fees_earned("Alice")
print(f"Fees immediately after deposit: {fees_earned}")

for _ in range(1000):
    token = random.choice(['x', 'y'])
    if token == 'x':
        amount = random.uniform(0, 0.02 * pool.reserve_x)
    else:
        amount = random.uniform(0, 0.02 * pool.reserve_y)
    pool.swap(amount, token)

fees_earned_after = pool.fees_earned("Alice")
print(f"Fees after 1000 swaps: {fees_earned_after}")

---

### 3.4 Experiment: Fee Income vs Volume

Simulate a trading session with many random swaps. Track cumulative LP earnings for each fee tier.

**Task 3.4** — Implement `simulate_trading`, then:
- For each fee tier (0.05%, 0.3%, 1.0%), create a pool with \$200k liquidity (50 ETH + 100,000 USDC)
- Simulate 2000 swaps, recording cumulative fee income every 50 swaps
- Plot all three tiers on the same chart

In [ ]:
def simulate_trading(pool: UniswapV2Pool, n_swaps: int = 1000, max_trade_pct: float = 0.02):
    """Simulate random swaps against the pool."""
    for _ in range(n_swaps):
        token = random.choice(['x', 'y'])
        max_amount = max_trade_pct * (pool.reserve_x if token == 'x' else pool.reserve_y)
        amount = random.uniform(0.0001 * max_amount, max_amount)
        pool.swap(amount, token)

In [ ]:
fee_tiers = [5, 30, 100]   # 0.05%, 0.3%, 1.0%
price_eth = 2000

for fee in fee_tiers:
    pool = UniswapV2Pool(0, 0, fee_bps=fee)
    pool.add_liquidity(50, 100_000, "Alice")
    
    fee_incomes = []
    swap_numbers = []
    for i in range(0, 2000, 50):
        simulate_trading(pool, n_swaps=50)
        fees = pool.fees_earned("Alice")
        fee_usd = (fees['fee_x'] * pool.price) + fees['fee_y']
        fee_incomes.append(fee_usd)
        swap_numbers.append(i + 50)
        
    plt.plot(swap_numbers, fee_incomes, label=f"{fee / 100}% Fee Tier")

plt.xlabel('Number of Swaps')
plt.ylabel('Cumulative Fee Income (USD)')
plt.title('Cumulative Fee Income by Fee Tier')
plt.legend()
plt.grid(True)
plt.show()

**Research questions (Task 3.4):**
- Which fee tier earns the most for a high-volume pair (e.g., ETH/USDC)? For a low-volume exotic pair?
- Real Uniswap V3 stablecoin pools (USDC/DAI) almost always use 0.05%. Why does your simulation support this?
- If volume doubles but pool TVL stays the same, how does APR change? Is the relationship linear?

YOUR ANSWER HERE

---

## Bonus: Your Own Experiment

Design and run one additional experiment exploring any parameter or behavior not covered above. Some ideas:
- How does impermanent loss vary with price drift?
- What happens with asymmetric trade flows (e.g., 80% buys, 20% sells)?
- Multi-hop routing: what if you need to trade A → B → C through two pools?
- Compare constant product ($xy=k$) with constant sum ($x+y=k$) — what breaks?

In [ ]:
# YOUR EXPERIMENT HERE


*Your explanation of the experiment and findings:*

YOUR ANSWER HERE

---

## Grading

| Criterion | Weight |
|---|---|
| Invariant correctness (`k` never decreases) | 20% |
| `get_amount_out` and `swap` math | 20% |
| LP share + fee accrual logic | 20% |
| Plots: correct, labeled, readable | 20% |
| Written analysis quality | 20% |

---

## Reference: Key Formulas

| Formula | Description |
|---|---|
| $x \cdot y = k$ | Constant product invariant |
| $\text{out} = \frac{y \cdot \text{in} \cdot (1-f)}{x + \text{in} \cdot (1-f)}$ | Amount out for X → Y swap |
| $\text{impact} = \frac{p_{\text{spot}} - p_{\text{exec}}}{p_{\text{spot}}}$ | Price impact |
| $\text{shares} = \frac{\Delta x}{x} \cdot S_{\text{total}}$ | LP shares minted |
| $\text{APR} \approx \frac{\text{fees\_per\_day}}{\text{deposit\_value}} \times 365$ | LP annual return estimate |